# FingerBall v2 — Stage 1: Drive上のWiLoR / 手検出モデルをColabで検証

**目的**: Google Driveの「母指回内トラッキング」に保存された既存の **OpenVINO INT8 WiLoR** と **手検出器**を使い、**フォルダ確認 → モデルの一時コピー → モデル入力出力検査 → 1フレーム手検出 → WiLoR推論 → 結果保存**を行います。GitHubへのアップロードは不要です。

- **GPUに関する重要事項**: この引き継ぎモデルはOpenVINO IR（XML/BIN）です。標準的なColabのNVIDIA GPUでCUDA推論する版ではなく、**CPU上で動作確認**します。WiLoRのPyTorch/CUDA版は後続段階で別途構築します。
- **本Stageの到達点**: モデルがロードでき、手検出が成立し、WiLoR推論の生出力が生成されること。**21ランドマーク・3D関節角度・指球接触はまだ算出しません。** それらは元プロジェクトの `to_params()`、MANO依存処理や新規ボール検出を移植するStage 2以降です。
- 元撮影fpsは不明でよく、フレーム番号で処理します。入力動画はDrive指定か手動アップロードから選択できます。
- 個人情報のある研究動画はColabへのアップロード権限・倫理上の扱いを確認してください。結果は指定したDriveフォルダに保存されます。


## 0. 環境準備

In [ ]:
%pip -q install "openvino>=2025.0,<2027" opencv-python-headless matplotlib pandas pyyaml

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 1. 設定 — 通常はここだけ編集

In [ ]:
from pathlib import Path
import os, json, time, shutil, hashlib, sys
import numpy as np
import cv2
import openvino as ov
import matplotlib.pyplot as plt
from IPython.display import display

# Driveフォルダを名前から探す。変更後の名称に対応。
PROJECT_FOLDER_NAME = '母指回内トラッキング'

# すでにMyDriveか共有ドライブの正確な絶対パスが分かる場合のみ設定。
DRIVE_PROJECT_DIR = ''  # 例: '/content/drive/MyDrive/母指回内トラッキング'

# 動画を使う場合はパスを指定。空欄ならColabから手動アップロードを選択。
VIDEO_PATH = ''
FRAME_INDEX = 0      # 0始まり。手が大きく写っているフレームへ変更
MANUAL_BOX = None    # YOLO手検出が失敗したとき (x0,y0,x1,y1) と指定可
RIGHT_HAND = True    # 検出クラスの選択。右手ならTrue（映像の左右とは別）

# Colabローカルへコピーしたキャッシュ
CACHE_ROOT = Path('/content/fingerball_stage1_models')
RESULTS_DIR_NAME = 'FingerBall_v2_stage1_results'


## 2. Drive内のフォルダを確認する

In [ ]:
def find_project(name, supplied=''):
    if supplied:
        q=Path(supplied)
        if not q.is_dir():
            raise FileNotFoundError(f'指定したDRIVE_PROJECT_DIRが存在しません: {q}')
        return q
    # mount領域の浅い階層から探す。Drive全体に対する再帰的rglobは遅いので避ける。
    roots=[Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), Path('/content/drive/共有ドライブ')]
    matches=[]
    for base in roots:
        if not base.is_dir():
            continue
        candidate=base/name
        if candidate.is_dir(): matches.append(candidate)
        if base.name!='MyDrive':
            try:
                for d in base.iterdir():
                    if d.is_dir() and (d/name).is_dir():matches.append(d/name)
            except OSError:pass
    if not matches:
        raise FileNotFoundError(f'「{name}」が見つかりません。Driveのマウント先を確認しDRIVE_PROJECT_DIRに絶対パスを指定してください。')
    if len(matches)>1:
        print('同名フォルダが複数あります。先頭を採用:', matches)
    return matches[0]

project=find_project(PROJECT_FOLDER_NAME, DRIVE_PROJECT_DIR)
print('対象フォルダ:',project)
print('直下の項目:',[x.name for x in project.iterdir()])
assert (project/'thumb_pronation').is_dir(), '想定構成と異なります。フォルダを確認してください。'


## 3. モデルを特定し、Colabローカルへコピー（Driveから直接推論しない）

In [ ]:
# 実際のGoogle Driveフォルダから確認された相対パス
WILOR_REL = Path('thumb_pronation/out/bench_light/verify/int8_leftcal')
DETECTOR_REL = Path('models/_DATA/wilor/pretrained_models/detector_openvino_model_288x512')

wilor_xml = project/WILOR_REL/'wilor_core_int8_leftcal.xml'
wilor_bin = wilor_xml.with_suffix('.bin')
detector_xml = project/DETECTOR_REL/'detector.xml'
detector_bin = detector_xml.with_suffix('.bin')
detector_meta = project/DETECTOR_REL/'metadata.yaml'

required = [wilor_xml,wilor_bin,detector_xml,detector_bin,detector_meta]
for x in required: print(('OK ' if x.is_file() else 'MISSING '),x)
missing=[str(x) for x in required if not x.is_file()]
if missing:
    raise FileNotFoundError('必要ファイルがありません。パスを確認:\n'+'\n'.join(missing))

CACHE_ROOT.mkdir(exist_ok=True,parents=True)
model_dir=CACHE_ROOT/'wilor';det_dir=CACHE_ROOT/'detector'
model_dir.mkdir(exist_ok=True);det_dir.mkdir(exist_ok=True)

def copy_if_changed(src,dstdir):
    dst=dstdir/src.name
    if not dst.exists() or dst.stat().st_size!=src.stat().st_size:
        print('コピー:',src.name,round(src.stat().st_size/1e6,1),'MB')
        shutil.copy2(src,dst)
    return dst

xml_local=copy_if_changed(wilor_xml,model_dir)
_ = copy_if_changed(wilor_bin,model_dir)
det_xml_local=copy_if_changed(detector_xml,det_dir)
_ = copy_if_changed(detector_bin,det_dir)
meta_local=copy_if_changed(detector_meta,det_dir)
print('コピー完了')


## 4. OpenVINOでモデルのロード・入力出力形状の検査

In [ ]:
core=ov.Core()
print('OpenVINO version:',ov.__version__)
print('Available OpenVINO devices:',core.available_devices)
# OpenVINOのGPUは一般的なColab NVIDIA CUDA GPUを意味しません
DEVICE='CPU'
start=time.perf_counter()
det_model=core.read_model(str(det_xml_local))
wilor_model=core.read_model(str(xml_local))

def describe(model,label):
    print('\n'+label)
    for i,p in enumerate(model.inputs):
        print(' INPUT',i,p.get_any_name(),str(p.partial_shape),str(p.element_type))
    for i,p in enumerate(model.outputs):
        print(' OUTPUT',i,p.get_any_name(),str(p.partial_shape),str(p.element_type))
describe(det_model,'HAND DETECTOR')
describe(wilor_model,'WILOR INT8')

print('モデルをCPUにコンパイル中...')
det_compiled=core.compile_model(det_model,DEVICE,{'PERFORMANCE_HINT':'LATENCY'})
wilor_compiled=core.compile_model(wilor_model,DEVICE,{'PERFORMANCE_HINT':'LATENCY'})
print('コンパイル完了 秒:',round(time.perf_counter()-start,1))


## 5. 動画または画像からフレームを取得

In [ ]:
from google.colab import files

if not VIDEO_PATH:
    print('ここで動画（mp4など）または1枚の写真を選択してください。')
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('入力ファイルが指定されませんでした')
    VIDEO_PATH=str(Path('/content')/next(iter(uploaded)))

source=Path(VIDEO_PATH)
if not source.exists():raise FileNotFoundError(source)
if source.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}:
    image=cv2.imread(str(source));source_frame=0
else:
    cap=cv2.VideoCapture(str(source))
    if not cap.isOpened():raise RuntimeError('OpenCVで動画を開けません')
    num_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('動画総フレーム数:',num_frames,'表示fps:',cap.get(cv2.CAP_PROP_FPS),'（元撮影fpsとは限りません）')
    cap.set(cv2.CAP_PROP_POS_FRAMES,FRAME_INDEX)
    ok,image=cap.read();cap.release()
    if not ok:raise RuntimeError('指定フレームが読み込めません。FRAME_INDEXを見直してください')
    source_frame=FRAME_INDEX
if image is None:raise RuntimeError('画像の読み込みに失敗')
H,W=image.shape[:2]
print('フレーム:',source_frame,'画像サイズ:',W,H)
plt.figure(figsize=(10,6));plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()


## 6. YOLO/OpenVINOで手の検出（検出器の推論とNMS）

In [ ]:
import yaml
meta=yaml.safe_load(meta_local.read_text(encoding='utf-8'))
print('モデルメタデータ:',meta)

# 当該引き継ぎモデルのYOLO姿勢検出器の形式に合わせた処理。
# 出力 (batch, 4+nc+nk, num_predictions) を想定。形式が異なる場合は明示的に止める。

def detector_input(image_bgr, net_h=288, net_w=512):
    h,w=image_bgr.shape[:2]
    gain=min(net_h/h,net_w/w)
    nw,nh=int(round(w*gain)),int(round(h*gain))
    left=(net_w-nw)/2;top=(net_h-nh)/2
    l=int(round(left-.1));r=int(round(left+.1));t=int(round(top-.1));btm=int(round(top+.1))
    small=cv2.resize(image_bgr,(nw,nh),interpolation=cv2.INTER_LINEAR)
    small=cv2.copyMakeBorder(small,t,btm,l,r,cv2.BORDER_CONSTANT,value=(114,114,114))
    x=np.ascontiguousarray(small[None,...,::-1].transpose(0,3,1,2),dtype=np.float32)/255.0
    return x,gain,(l,t)

def nms_xyxy(boxes,scores,iou_thr=.7):
    if len(boxes)==0:return np.array([],dtype=int)
    x1,y1,x2,y2=boxes.T
    area=np.maximum(0,x2-x1)*np.maximum(0,y2-y1)
    order=np.argsort(-scores);keep=[]
    while len(order):
        i=order[0];keep.append(int(i));rest=order[1:]
        if not len(rest):break
        iw=np.maximum(0,np.minimum(x2[i],x2[rest])-np.maximum(x1[i],x1[rest]))
        ih=np.maximum(0,np.minimum(y2[i],y2[rest])-np.maximum(y1[i],y1[rest]))
        overlap=iw*ih/(area[i]+area[rest]-iw*ih+1e-12)
        order=rest[overlap<=iou_thr]
    return np.asarray(keep,dtype=int)

shape=list(det_compiled.input(0).shape)
assert len(shape)==4 and shape[0]==1 and shape[1]==3, f'未対応の手検出器入力: {shape}'
net_h,net_w=int(shape[2]),int(shape[3])
det_x,gain,(pad_x,pad_y)=detector_input(image,net_h,net_w)
start=time.perf_counter()
det_raw=np.asarray(det_compiled([det_x])[det_compiled.output(0)])
print('推論時間:',round(time.perf_counter()-start,2),'s','出力形状:',det_raw.shape)

nc=len(meta.get('names',{0:'left',1:'right'}))
if det_raw.ndim!=3 or det_raw.shape[0]!=1:
    raise RuntimeError(f'検出器出力の解釈が必要: {det_raw.shape}')
pred=det_raw[0]
if pred.shape[0]<4+nc and pred.shape[1]>=4+nc:pred=pred.T
if pred.shape[0]<4+nc:
    raise RuntimeError(f'推定器出力がYOLO形式と一致しません: {pred.shape}')
pred=pred.T
scores_all=pred[:,4:4+nc]
conf=scores_all.max(axis=1);classes=scores_all.argmax(axis=1)
keep=conf>=.3
p,conf,classes=pred[keep],conf[keep],classes[keep]
if len(p):
    xywh=p[:,:4];boxes=np.column_stack([xywh[:,0]-xywh[:,2]/2,xywh[:,1]-xywh[:,3]/2,xywh[:,0]+xywh[:,2]/2,xywh[:,1]+xywh[:,3]/2])
    chosen=nms_xyxy(boxes+classes[:,None]*7680,conf)[:300]
    boxes,conf,classes=boxes[chosen],conf[chosen],classes[chosen]
    boxes[:,[0,2]]=(boxes[:,[0,2]]-pad_x)/gain
    boxes[:,[1,3]]=(boxes[:,[1,3]]-pad_y)/gain
    boxes[:,[0,2]]=boxes[:,[0,2]].clip(0,W)
    boxes[:,[1,3]]=boxes[:,[1,3]].clip(0,H)
else:boxes=np.zeros((0,4));conf=np.zeros(0);classes=np.zeros(0,int)
print('検出数:',len(boxes),'（右/左のラベルはメタデータ・モデル実装と照合）')
for i,(box,score,cl) in enumerate(zip(boxes,conf,classes)):
    print(i,'class=',int(cl),'score=',round(float(score),3),'box=',np.rint(box).astype(int))
vis=image.copy()
for i,(box,score,cl) in enumerate(zip(boxes,conf,classes)):
    x0,y0,x1,y1=np.rint(box).astype(int)
    cv2.rectangle(vis,(x0,y0),(x1,y1),(0,210,0),2)
    cv2.putText(vis,f'{i}: {int(cl)} {score:.2f}',(x0,max(15,y0-5)),cv2.FONT_HERSHEY_SIMPLEX,.55,(255,0,0),2)
plt.figure(figsize=(12,6));plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()


## 7. 手領域を決めてWiLoR INT8で1回推論する

モデルの生出力を得る段階です。**関節座標への復元はまだ実装していません。** 貼り付け位置や右左が違うと正しく推定されません。

In [ ]:
# YOLOクラスの意味を確認し、対象側の検出を選択。
print('YOLO names:',meta.get('names'))
labels=meta.get('names',{})

def class_name(cls):
    return str(labels.get(int(cls),labels.get(str(int(cls)),str(cls)))).lower()

if MANUAL_BOX is not None:
    box=np.asarray(MANUAL_BOX,float)
else:
    # 既存wilor_runtime.pyでは右手 class=1、左手 class=0
    desired=1 if RIGHT_HAND else 0
    idx=np.where(classes==desired)[0]
    if len(idx)==0:
        raise RuntimeError('投球手の検出がありません。FRAME_INDEXを変更するかMANUAL_BOX=(x0,y0,x1,y1)を指定してください')
    pick=idx[np.argmax(conf[idx])]
    box=np.asarray(boxes[pick],float)
print('採用BOX:',box,' right:',RIGHT_HAND)


In [ ]:
# WiLoR既存実装のmake_patchと同じ領域拡張(2.5倍)・左手反転・256x256
# フィルタ計算までを完全一致させるためには元コードwilor_runtime.make_patch_fastが必要です。
# 本Stageは「モデル読み込みとforwardの検証」であり、前処理・出力の数値互換性検証ではありません。

def make_smoke_patch(image_bgr,box,right=True,out_size=256,rescale=2.5):
    h,w=image_bgr.shape[:2]
    cx,cy=(box[:2]+box[2:])/2
    side=float(max(box[2]-box[0],box[3]-box[1])*rescale)
    im=image_bgr if right else np.ascontiguousarray(image_bgr[:,::-1])
    if not right:cx=w-1-cx
    x0,y0=int(np.floor(cx-side/2)),int(np.floor(cy-side/2))
    x1,y1=int(np.ceil(cx+side/2)),int(np.ceil(cy+side/2))
    pad_l,pad_t=max(-x0,0),max(-y0,0)
    pad_r,pad_b=max(x1-w,0),max(y1-h,0)
    if pad_l or pad_r or pad_t or pad_b:
        im=cv2.copyMakeBorder(im,pad_t,pad_b,pad_l,pad_r,cv2.BORDER_CONSTANT,value=(0,0,0))
    patch=im[y0+pad_t:y1+pad_t,x0+pad_l:x1+pad_l]
    if patch.size==0:raise ValueError('手領域の切り出しに失敗しました')
    patch=cv2.resize(patch,(out_size,out_size),interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return patch

input_shape=list(wilor_compiled.input(0).shape)
print('WiLoR入力:',input_shape)
assert len(input_shape)==4 and input_shape[0]==1, 'WiLoR入力形式に対応していません'
patch=make_smoke_patch(image,box,RIGHT_HAND,256)
# 訓練時のOpenVINO版が想定するtensor layoutに従う
if input_shape[1]==3:
    model_x=patch.transpose(2,0,1)[None]
elif input_shape[-1]==3:
    model_x=patch[None]
else:raise RuntimeError('予期しない入力shape: '+str(input_shape))
if tuple(model_x.shape)!=tuple(input_shape):
    raise RuntimeError(f'入力不一致: generated {model_x.shape}, required {input_shape}')
plt.figure(figsize=(4,4));plt.imshow(cv2.cvtColor(patch.astype(np.uint8),cv2.COLOR_BGR2RGB));plt.title('WiLoR input patch (smoke test)');plt.axis('off');plt.show()
start=time.perf_counter()
raw=wilor_compiled([np.ascontiguousarray(model_x)])
elapsed=time.perf_counter()-start
output_map={}
for i,port in enumerate(wilor_compiled.outputs):
    ar=np.asarray(raw[port]);output_map[f'out_{i}']=ar
    print('output',i,'name=',port.get_any_name(),'shape=',ar.shape,'finite=',bool(np.all(np.isfinite(ar))))
print('WiLoR CPU forward:',round(elapsed,2),'秒')


## 8. 結果の保存（Drive側は成果物のみ）

In [ ]:
from datetime import datetime
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
result_dir=project/RESULTS_DIR_NAME/stamp
result_dir.mkdir(parents=True,exist_ok=False)
cv2.imwrite(str(result_dir/'source_frame.jpg'),image)
cv2.imwrite(str(result_dir/'hand_detection.jpg'),vis)
cv2.imwrite(str(result_dir/'wilor_patch.jpg'),patch.astype(np.uint8))
np.savez_compressed(str(result_dir/'wilor_raw_outputs.npz'),**output_map)
report={
  'source_name':source.name,'frame_index':int(source_frame),'image_wh':[W,H],
  'opencv_version':cv2.__version__,'openvino_version':ov.__version__,
  'device':DEVICE,'hand_right':RIGHT_HAND,'manual_box':MANUAL_BOX,
  'selected_box':list(map(float,box)),'detector_boxes':np.asarray(boxes).tolist(),
  'detector_scores':np.asarray(conf).tolist(),'detector_classes':np.asarray(classes).astype(int).tolist(),
  'wilor_output_shapes':{k:list(v.shape) for k,v in output_map.items()},
  'wilor_forward_seconds':round(elapsed,3),
  'warning':'Smoke test only: approximate WiLoR patch; no 21-point projection or angle validation.'
}
(result_dir/'stage1_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('保存先:',result_dir)
print('ファイル:',[x.name for x in result_dir.iterdir()])


## 判定基準と次の段階

- **PASS**: Driveが見つかる、5つのモデルファイルが揃う、両モデルがコンパイルできる、WiLoRの出力が有限。
- **追加確認**: 正しい手がYOLOで検出されたか、指が入力パッチ内に完全に写っているか。
- **未検証**: `wilor_runtime.py` と一致する厳密なpatch処理／WiLoRパラメータ変換／MANO→21点投影／手指角度精度／ボール追跡。今回のStage 1は、それらを実装したと主張しません。
- **Stage 2**: 既存の `tools/bench_wilor/wilor_ov_params.py` の `make_patch/to_params` と `appB/wilor_runtime.py` の `project_keypoints` を使い、21点・3Dメッシュを復元する。その後、球検出とリリース解析を統合します。

**ライセンス**: モデル重みとMANO・WiLoRの取り扱いは既存READMEに従ってください。パブリックGitHubへの再配布は避けてください。